# Tokenize SFT Dataset
Reads raw JSONL from `datasets_sft/<dataset>/`, tokenizes with the pretrain tokenizer,
and writes `<dataset>_tokenized.jsonl` with `input_ids` and `prefix_len` for masked SFT training.

Run `download_alpaca` first. Run `sft_alpaca` after.

In [ ]:
import os
import json
from google.colab import drive
from tokenizers import Tokenizer

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")

# ==================== CONFIGURATION ====================
DATASET_NAME = "alpaca"  # change this for other SFT datasets
BLOCK_SIZE = 768
MIN_RESPONSE_TOKENS = 3

SFT_DIR = os.path.join(SPARKYLLM, "datasets_sft", DATASET_NAME)
RAW_FILE = os.path.join(SFT_DIR, f"{DATASET_NAME}_raw.jsonl")
OUT_FILE = os.path.join(SFT_DIR, f"{DATASET_NAME}_tokenized.jsonl")
META_FILE = os.path.join(SFT_DIR, "tokenize_meta.json")

assert os.path.exists(RAW_FILE), f"Raw file not found: {RAW_FILE}\nRun download_{DATASET_NAME} first."

# Load tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
eot_id = tokenizer.token_to_id("<|endoftext|>")
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer: {vocab_size} tokens | EOT: {eot_id}")
print(f"Raw file: {RAW_FILE}")

In [ ]:
# ==================== FORMAT ADAPTER ====================
# Each SFT dataset needs a function that takes a raw JSON object
# and returns (prefix_text, response_text).
# Add new adapters here for other datasets.

def alpaca_adapter(obj):
    """Alpaca format: instruction + optional input -> response."""
    parts = [f"Instruction: {obj['instruction']}"]
    if obj.get("input"):
        parts.append(f"Input: {obj['input']}")
    parts.append("Response:")
    prefix_text = "\n".join(parts) + " "
    response_text = obj["output"]
    return prefix_text, response_text

ADAPTERS = {
    "alpaca": alpaca_adapter,
}

adapter = ADAPTERS[DATASET_NAME]
print(f"Using adapter: {DATASET_NAME}")

In [ ]:
# ==================== TOKENIZE ====================

written = 0
skipped = 0
total_tokens = 0
total_prefix_len = 0

with open(OUT_FILE, "w", encoding="utf-8") as f_out:
    with open(RAW_FILE, "r", encoding="utf-8") as f_in:
        for line_num, line in enumerate(f_in):
            obj = json.loads(line)
            prefix_text, response_text = adapter(obj)

            prefix_ids = tokenizer.encode(prefix_text).ids
            response_ids = tokenizer.encode(response_text).ids

            if len(response_ids) < MIN_RESPONSE_TOKENS:
                skipped += 1
                continue

            # Full sequence: prefix + response + EOT
            full_ids = prefix_ids + response_ids + [eot_id]

            # Truncate to block_size + 1 (need +1 for shifted labels)
            if len(full_ids) > BLOCK_SIZE + 1:
                full_ids = full_ids[:BLOCK_SIZE + 1]

            out_obj = {
                "input_ids": full_ids,
                "prefix_len": len(prefix_ids),
            }
            f_out.write(json.dumps(out_obj) + "\n")

            written += 1
            total_tokens += len(full_ids)
            total_prefix_len += len(prefix_ids)

            if (line_num + 1) % 10000 == 0:
                print(f"  {line_num + 1:,} lines processed...")

avg_prefix = total_prefix_len / written if written else 0
avg_total = total_tokens / written if written else 0

print(f"\nDone!")
print(f"  {written:,} examples written, {skipped:,} skipped")
print(f"  {total_tokens:,} total tokens")
print(f"  Avg prefix: {avg_prefix:.1f} tokens | Avg total: {avg_total:.1f} tokens")
print(f"  Saved to: {OUT_FILE}")

# Save metadata
meta = {
    "dataset": DATASET_NAME,
    "num_examples": written,
    "skipped": skipped,
    "total_tokens": total_tokens,
    "avg_prefix_len": round(avg_prefix, 1),
    "avg_total_len": round(avg_total, 1),
    "block_size": BLOCK_SIZE,
    "eot_id": eot_id,
    "vocab_size": vocab_size,
    "tokenizer_path": TOKENIZER_PATH,
}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)
print(f"  Meta: {META_FILE}")